# บทที่ 7 · ดึงข้อมูลจาก API

ร้านของเรา (บทที่ 3) จ่ายเงิน **ค่าโฆษณาออนไลน์** ทุกวัน ดูยอดได้จาก **API** ของระบบโฆษณา
API = หน้าเว็บสำหรับโปรแกรม เป็นแหล่งข้อมูลแบบที่ 3 ต่อจากไฟล์ CSV และฐานข้อมูล

API ที่ใช้: เปิดด้วย Docker แล้ว (ดู README) เอกสารอยู่ที่ http://localhost:8000/docs
**ตั้งใจให้ error สุ่ม 25%** เหมือนเว็บจริงที่บางครั้งไม่ว่าง

บทนี้: **requests · status code · API key · params · หลายหน้า · ลองใหม่ · ผลจาก API → DataFrame**

In [ ]:
import os

import requests
from dotenv import load_dotenv

load_dotenv()
API_URL = os.getenv("SHOP_API_URL", "http://127.0.0.1:8000")
API_KEY = os.getenv("SHOP_API_KEY", "class-demo-key")
print(API_URL)

## 7.1 ขอข้อมูลจากเว็บด้วย `requests.get`

API = หน้าเว็บสำหรับโปรแกรม: เราส่งคำขอ (request) ไป แล้วได้คำตอบ (response) กลับมาเป็น JSON

In [ ]:
response = requests.get(f"{API_URL}/health", timeout=10)     # timeout: ไม่รอเกิน 10 วินาที

print(response.status_code)     # 200 = สำเร็จ
print(response.text)            # คำตอบแบบข้อความ
print(response.json())          # แปลง JSON -> dict

| status | ความหมาย | ทำอย่างไร |
|---|---|---|
| 200 | สำเร็จ | ใช้ข้อมูล |
| 400 / 422 | เราส่งค่าผิด | แก้โค้ด (ลองใหม่ไม่ช่วย) |
| 401 / 403 | ไม่มีสิทธิ์ / key ผิด | แก้ key (ลองใหม่ไม่ช่วย) |
| 404 | ไม่มีหน้านี้ | ตรวจ URL |
| 429 | ขอถี่เกินไป | รอแล้วลองใหม่ |
| 500 / 503 | ฝั่งเว็บมีปัญหา | รอแล้วลองใหม่ |

## 7.2 API key ใส่ใน headers และ `raise_for_status()`

In [ ]:
no_key = requests.get(f"{API_URL}/v1/campaigns", timeout=10)
print(no_key.status_code, no_key.json())

In [ ]:
no_key.raise_for_status()      # ❌ status ไม่ใช่ 2xx -> raise HTTPError ทำให้รู้ตัวทันที

In [ ]:
headers = {"X-API-Key": API_KEY}
campaigns = requests.get(f"{API_URL}/v1/campaigns", headers=headers, timeout=10)
campaigns.raise_for_status()                    # ผ่าน = ไม่มีอะไรเกิดขึ้น
for c in campaigns.json()["data"]:
    print(c["campaign_id"], "|", c["channel"], "|", c["campaign_name"])

## 7.3 ส่งเงื่อนไขด้วย `params` และรับมือเมื่อเว็บไม่ว่าง

`params={"date": "2026-09-25"}` → requests ต่อท้าย URL ให้เป็น `?date=2026-09-25` เอง

🧪 กดรัน cell ข้างล่างซ้ำหลาย ๆ รอบ จะเห็น 429 / 503 บ้างเป็นบางครั้ง

In [ ]:
r = requests.get(f"{API_URL}/v1/ad-spend", params={"date": "2026-09-25", "page": 1}, headers=headers, timeout=10)
print(r.url)
print(r.status_code, r.text[:200])

In [ ]:
import time


def get_json(path: str, params: dict | None = None, max_tries: int = 5) -> dict:
    """ขอข้อมูลจาก API แล้วคืน dict · ลองใหม่เฉพาะเมื่อเว็บไม่ว่าง (429/5xx) · ปัญหาอื่นหยุดทันที"""
    for attempt in range(1, max_tries + 1):
        resp = requests.get(f"{API_URL}{path}", params=params, headers=headers, timeout=10)
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_tries:
            wait = int(resp.headers.get("Retry-After", 1))       # เว็บบอกว่าให้รอกี่วินาที
            print(f"  ครั้งที่ {attempt}: HTTP {resp.status_code} รอ {wait} วินาที")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()


body = get_json("/v1/ad-spend", {"date": "2026-09-25", "page": 1})
print({k: body[k] for k in ["date", "page", "page_size", "total_pages", "total_records"]})

## 7.4 ข้อมูลหลายหน้า (pagination)

API ไม่ส่งข้อมูลทั้งหมดในครั้งเดียว ต้องขอทีละหน้า `page = 1, 2, 3, ...` จนถึง `total_pages`

In [ ]:
def fetch_ad_spend(day: str) -> list[dict]:
    """ค่าโฆษณาทุกแคมเปญของวันนั้น (ขอทีละหน้าจนครบ)"""
    rows, page = [], 1
    while True:
        body = get_json("/v1/ad-spend", {"date": day, "page": page})
        rows.extend(body["data"])            # extend = เอาทุกแถวของหน้านี้ต่อท้าย rows
        if page >= body["total_pages"]:
            break
        page += 1
    print(f"{day}: ได้ {len(rows)} แถว จาก {page} หน้า")
    return rows


ad_rows = fetch_ad_spend("2026-09-25")
ad_rows[:2]

## 7.5 ผลจาก API → DataFrame (🔁 บทที่ 3)

ข้อมูลจาก API เป็น list ของ dict → `pd.DataFrame(...)` ได้ตารางทันที

In [ ]:
import pandas as pd

ads = pd.DataFrame(ad_rows)
print(ads.shape)
print("ค่าโฆษณารวม:", ads["spend"].sum(), "บาท")
ads[["campaign_name", "channel", "clicks", "spend"]].sort_values("spend", ascending=False).head()

## 7.6 ยอดขาย marketplace เทียบกับค่าโฆษณา

ค่าโฆษณา 1 บาท ได้ยอดขายกลับมากี่บาท (เตรียมยอดขายแบบเดียวกับบทที่ 3.4)

In [ ]:
mkt = pd.read_csv("../data/marketplace/marketplace_orders_2026-09-25.csv")
mkt = mkt[(mkt["status"] != "cancelled") & (mkt["qty"] > 0)]
revenue = float((mkt["qty"] * mkt["unit_price"]).sum())
spend = float(ads["spend"].sum())
print(f"2026-09-25 ยอดขาย marketplace {revenue:,.2f} บาท · ค่าโฆษณา {spend:,.2f} บาท · ได้คืน {revenue / spend:.1f} เท่า")

## ✅ สรุปบทที่ 7

| เรื่อง | ตัวอย่าง |
|---|---|
| ขอข้อมูล | `requests.get(url, params={...}, headers={...}, timeout=10)` |
| ตรวจผล | `.status_code`, `.json()`, `.raise_for_status()` |
| ลองใหม่ | ลองใหม่เฉพาะ 429 / 5xx (เว็บไม่ว่าง) ปัญหาอื่นหยุดทันที |
| หลายหน้า | `while True` + `page += 1` + `break` |
| เป็นตาราง | `pd.DataFrame(rows)` |

🧪 **แบบฝึกท้ายบท:** ดึงค่าโฆษณา 3 วัน (`2026-09-23` ถึง `2026-09-25`) รวมเป็นตารางเดียว
แล้วหา **ค่าโฆษณารวมต่อช่องทาง (channel)** เรียงจากมากไปน้อย

In [ ]:
# เขียนต่อตรงนี้